# IMEN266 · HW 5 workspace (Ch.8)

Statement: `hw5.pdf` (PLMS / repo). This notebook is your **computational
workspace** for Parts B–C and your **prompt log** for Part D.
Part A is pen-and-paper first — no cells here on purpose.

▶ Colab: `https://colab.research.google.com/github/youngmko/imen266-2026/blob/main/ch08/homework/HW5.ipynb`

In [ ]:
import math, heapq
import numpy as np
import matplotlib.pyplot as plt
try:
    from imen266.queueing import (mm1, mms, mm1k, mmsk, mg1_pk, jackson_solve, braess,
                                  simulate_queue, sim_network)
except ImportError:                                  # Colab: fetch the course package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/youngmko/imen266-2026.git"], check=True)
    from imen266.queueing import (mm1, mms, mm1k, mmsk, mg1_pk, jackson_solve, braess,
                                  simulate_queue, sim_network)
rng = np.random.default_rng()      # your results should NOT depend on a seed
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": .3})

def ci95(x):
    x = np.asarray(x, dtype=float)
    return 1.96 * x.std(ddof=1) / math.sqrt(len(x)) if len(x) > 1 else np.nan

---
## Part B2 — Correct numbers (formulas on paper, numbers here)

Desk: $M/M/1/4$ with $\lambda=10$/h, $\mu_1=8$/h. Check: a deterministic 5-minute
machine fed by the desk's output. Fill the desk numbers by hand (they are exact);
for the check, compute the two candidate approximations and say which hypothesis
each one needs.

In [ ]:
# --- B2 skeleton -------------------------------------------------------------
lam, mu1, K = 10.0, 8.0, 4                  # per hour
rho1 = lam / mu1                            # 1.25 -- and yet the desk is perfectly stable (why?)
p    = np.full(K + 1, np.nan)               # TODO: stationary distribution of M/M/1/4 (rho != 1 formula)
pK   = np.nan                               # TODO: fraction turned away
lam_eff = np.nan                            # TODO: rate of people who actually enter (per hour)
L1   = np.nan                               # TODO: mean number at the desk
W1   = np.nan                               # TODO: mean time at the desk for those who enter (minutes)
Lq1  = np.nan                               # TODO: mean number waiting (two ways: L - busy fraction, or lam_eff * Wq)

if np.isnan([pK, lam_eff, L1, W1, Lq1]).any():
    print("fill p, pK, lam_eff, L1, W1, Lq1 first")
else:
    print(f"p = {np.round(p, 4)}   loss {pK:.4f}   lam_eff = {lam_eff:.3f}/h   L = {L1:.3f}   W = {W1:.2f} min   Lq = {Lq1:.3f}   busy {1 - p[0]:.4f} = {lam_eff/mu1:.4f}")

# the check (stage 2): a constant 5-minute service. Two candidate formulas -- neither is exact here (why?)
lam2 = lam_eff / 60 if not np.isnan(lam_eff) else np.nan             # per minute
if not np.isnan(lam2):
    print(f"stage 2: rho2 = {lam2*5:.4f}")
    print(f"  Pollaczek-Khinchine with constant service (needs Poisson input):   Wq2 = {mg1_pk(lam2, 5.0, 0.0)['Wq']:.3f} min")
    print(f"  the AI's single-server formula rho/(mu(1-rho)) (that is M/M/1!):     Wq2 = {mm1(lam2, 1/5)['Wq']:.3f} min")
# TODO: which of the two is closer to the truth, and in which direction will even the P-K value err? (Part B3 tells.)

---
## Part B3 — Simulate the two-stage system directly (no queueing formula inside)

Event-driven: Poisson arrivals, an $\exp(\mu_1)$ clerk with room for $K$, a constant
5-minute check behind it. One wait per customer; losses counted at the door; the
inter-departure times of the desk are recorded to test the "Poisson output" claim.

In [ ]:
# --- B3 skeleton -------------------------------------------------------------
def simulate_two_stage(T, rng, lam=10/60, mu1=8/60, K=4, s2=5.0):
    """Stage 1: M/M/1/K (rates per minute); everyone served at stage 1 proceeds to stage 2: one machine, constant s2.
    Returns loss fraction, L1, W1, Wq1 (entering customers), L2, W2, Wq2, and cv / lag-1 correlation of the desk's output gaps."""
    n1 = n2 = 0; busy1 = busy2 = False; q1 = []; q2 = []
    ev = [(rng.exponential(1/lam), 0, 0)]                     # (time, kind, customer): 0 arrival, 1 desk done, 2 check done
    cid = 0; arrivals = 0; lost = 0
    t_in1, t_in2 = {}, {}
    w1, wq1, w2, wq2, dep1 = [], [], [], [], []
    area1 = area2 = 0.0; last = 0.0
    while ev:
        t, kind, c = heapq.heappop(ev)
        if t > T:
            break
        area1 += n1 * (t - last); area2 += n2 * (t - last); last = t
        if kind == 0:                                           # arrival at the desk
            heapq.heappush(ev, (t + rng.exponential(1/lam), 0, 0)); arrivals += 1
            if n1 == K:
                lost += 1; continue                             # turned away
            cid += 1; n1 += 1; t_in1[cid] = t
            if not busy1:
                busy1 = True; wq1.append(0.0); heapq.heappush(ev, (t + rng.exponential(1/mu1), 1, cid))
            else:
                q1.append(cid)
        elif kind == 1:                                         # desk done -> stage 2
            n1 -= 1; w1.append(t - t_in1.pop(c)); dep1.append(t)
            if q1:
                c2 = q1.pop(0); wq1.append(t - t_in1[c2]); heapq.heappush(ev, (t + rng.exponential(1/mu1), 1, c2))
            else:
                busy1 = False
            n2 += 1; t_in2[c] = t
            if not busy2:
                busy2 = True; wq2.append(0.0); heapq.heappush(ev, (t + s2, 2, c))
            else:
                q2.append(c)
        else:                                                   # check done
            n2 -= 1; w2.append(t - t_in2.pop(c))
            if q2:
                c2 = q2.pop(0); wq2.append(t - t_in2[c2]); heapq.heappush(ev, (t + s2, 2, c2))
            else:
                busy2 = False
    g = np.diff(np.array(dep1))
    return dict(loss=lost / arrivals, L1=area1 / last, W1=np.mean(w1), Wq1=np.mean(wq1),
                L2=area2 / last, W2=np.mean(w2), Wq2=np.mean(wq2),
                cv_out=g.std() / g.mean(), lag1_out=np.corrcoef(g[:-1], g[1:])[0, 1])

reps = 5
runs = [simulate_two_stage(200_000, rng) for _ in range(reps)]      # 200,000 minutes each (about 140 days)
for k in runs[0]:
    v = np.array([r[k] for r in runs])
    print(f"{k:>9}: {v.mean():8.4f} +/- {ci95(v):.4f}   (95% CI over {reps} runs)")
# TODO: compare loss, L1, W1 with your B2 numbers and with the AI's Steps 2-3; compare Wq2 with both formulas of B2.
# TODO: cv_out and lag1_out -- is the desk's output a Poisson process? What does that do to the P-K value for stage 2?

---
## Part C1 — Run length near saturation (prediction first!)

$L=\rho/(1-\rho)$ is exact, but the queue's correlation time grows like
$1/(1-\rho)^2$: the effective number of independent observations in a run of
length $T$ shrinks accordingly.

**C1 prediction (before running!):** how does the relative error of $\hat L$ grow
with $\rho$ at fixed $T$, and how must $T$ grow to keep it at $\pm5\%$? ___

In [ ]:
# --- C1 skeleton -------------------------------------------------------------
T, reps = 20_000, 10
print(f"{'rho':>5} {'L exact':>8} {'L hat':>8} {'95% CI':>8} {'rel. error':>10} {'T for +-5%':>11}")
for rho in [0.5, 0.7, 0.9, 0.95, 0.98]:
    Ls = np.array([simulate_queue(rho, 1.0, T=T, rng=rng, warmup=200)["L"] for _ in range(reps)])
    exact = rho / (1 - rho)
    half = ci95(Ls)
    T_need = T * (half / (0.05 * exact))**2                      # CI half-width scales like 1/sqrt(T)
    print(f"{rho:>5} {exact:>8.2f} {Ls.mean():>8.2f} {half:>8.2f} {half/exact:>10.1%} {T_need:>11.0f}")
# TODO: compare the growth of 'T for +-5%' with your prediction; which rho would you refuse to estimate from one day of data?
# NOTE: at rho = 0.98 the estimate is also BIASED low (the run starts empty and the warm-up is short) -- a confidence interval
#       measures noise, not bias. How would you detect the bias, and what would you change?

---
## Part C2 — Where the product form breaks (Bay-Gull Bagels)

Stations: Bagels (3 servers, 2.5 min), Smoothies (2, 4.5), Drinks (self-service, 2),
Cashier (2, 1), Eat in (self-service, 20); external rates $(1,0.2,0.4,0,0)$ per minute;
routing as in CP13 §2.

In [ ]:
# --- C2 skeleton -------------------------------------------------------------
names = ["Bagels", "Smoothies", "Drinks", "Cashier", "Eat in"]
theta = np.array([1.0, 0.2, 0.4, 0.0, 0.0]); mu = 1 / np.array([2.5, 4.5, 2.0, 1.0, 20.0]); s = [3, 2, np.inf, 2, np.inf]
P = [[0, .2, .3, .5, 0], [0, 0, 0, 1, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, .5], [0, 0, .05, 0, 0]]
ref = jackson_solve(theta, P, mu, s)
det = lambda m: (lambda r, n: np.full(n, m))
configs = {"(a) all exponential":              None,
           "(b) deterministic Smoothies":      [None, det(4.5), None, None, None],
           "(c) det. Smoothies + det. Cashier": [None, det(4.5), None, det(1.0), None]}
results = {}
for name, svc in configs.items():
    L = np.array([sim_network(theta, P, mu, s, T=50_000, rng=rng, warmup=1_000, service=svc)["L_i"] for _ in range(4)])
    results[name] = L
print(f"{'station':>10} {'product form':>13}" + "".join(f" {k:>36}" for k in configs))
for i, nm in enumerate(names):
    print(f"{nm:>10} {ref['L_i'][i]:>13.2f}" + "".join(f" {results[k][:, i].mean():>26.2f} +- {ci95(results[k][:, i]):<6.2f}" for k in configs))
print(f"{'store':>10} {ref['L']:>13.2f}" + "".join(f" {results[k].sum(1).mean():>26.2f} +- {ci95(results[k].sum(1)):<6.2f}" for k in configs))
# TODO: station by station -- which numbers move beyond their error bars, and why do the others stay?
#       (upstream / M/G/infinity insensitivity / non-Poisson input)

**C2 write-up (≤5 sentences, the two general conditions for a product form; mark what you could not verify):**

> ...

---
## Part C3 — Braess by simulation

Fixed delays are infinite-server stations with deterministic service. Before the
link: half of $\PP(2\lambda)$ enters at queue $x$ (then delay 2), half at the delay
(then queue $y$). After the link, at the Wardrop equilibrium of the lecture
($x=y=1$): queue flows $\mu-1$, middle-route flow $2(\mu-1)-2\lambda$.

In [ ]:
# --- C3 skeleton -------------------------------------------------------------
delay = lambda d: (lambda r, n: np.full(n, d))

def sim_before(lam, mu, T=30_000):
    P = np.zeros((4, 4)); P[0, 1] = 1; P[2, 3] = 1              # x -> delay2 ; delay2 -> y
    return sim_network([lam, 0, lam, 0], P, [mu, 1, 1, mu], [1, np.inf, np.inf, 1], T=T, rng=rng, warmup=1_000,
                       service=[None, delay(2.0), delay(2.0), None])["W"]

def sim_after(lam, mu, T=30_000):
    """Routing probabilities from the equilibrium flows of braess(); falls back to 'link unused' / 'middle only' regimes."""
    f_up, f_mid, f_low = braess(lam, mu)["flows"]
    into_x = f_up + f_mid
    P = np.zeros((5, 5)); P[2, 3] = 1; P[4, 3] = 1                 # delay2 -> y ; link -> y
    if into_x > 0:
        P[0, 1] = f_up / into_x; P[0, 4] = f_mid / into_x           # x -> delay2 or x -> link
    return sim_network([into_x, 0, f_low, 0, 0], P, [mu, 1, 1, mu, 1], [1, np.inf, np.inf, 1, np.inf], T=T, rng=rng,
                       warmup=1_000, service=[None, delay(2.0), delay(2.0), None, delay(1.0)])["W"]

b = braess(1.0, 2.5)
print(f"lam = 1, mu = 2.5:  before {sim_before(1.0, 2.5):.3f} (theory {b['W_before']:.3f})   after {sim_after(1.0, 2.5):.3f} (theory {b['W_after']:.3f})")

mus = [1.5, 1.8, 2.0, 2.3, 2.5, 2.8, 3.0, 3.2, 3.4, 3.6, 4.0]
print(f"{'mu':>4} {'before (sim)':>13} {'after (sim)':>12} {'regime':>14}   analytic thresholds: link hurts for lam+1 = 2 < mu < 2+sqrt(2) = {2+math.sqrt(2):.3f}")
for m in mus:
    print(f"{m:>4} {sim_before(1.0, m, T=20_000):>13.3f} {sim_after(1.0, m, T=20_000):>12.3f} {braess(1.0, m)['regime']:>14}")
# TODO: where does the link hurt empirically? compare with lam + 1 and 2 + sqrt(2).

# planner: force a fraction f of the demand through the link (rest split evenly), mu = 2.5, lam = 1
f = np.linspace(0, 1.99, 200); fq = (2 + f) / 2                        # flow through each queue
xq = 1 / (2.5 - fq)                                                     # M/M/1 sojourn at each queue
W_avg = ((2 - f) * (xq + 2) + f * (2 * xq + 1)) / 2
plt.figure(); plt.plot(f, W_avg); plt.axhline(3.0, color="r", ls=":", label="selfish equilibrium W = 3")
plt.xlabel("flow forced through the link"); plt.ylabel("mean sojourn of everybody"); plt.legend(); plt.title("What a planner would do"); plt.show()
# TODO: state the planner's decision; (optional) confirm one point of the curve by simulation with sim_network

---
## Part C4 (optional) — An input that is not Poisson (feedback queue)

In [ ]:
# --- C4 skeleton -------------------------------------------------------------
lam_f, mu_f = 1.0, 4.0
for q in [0.0, 0.3, 0.6]:
    r = sim_network([lam_f], [[q]], [mu_f], T=30_000, rng=rng, warmup=500, record=True)
    gaps = np.diff(r["arrivals_at"][0]); gaps = gaps[gaps > 0]
    t, n = r["t"], r["n_path"][:, 0]
    grid = np.arange(500, 30_000, 0.01); X = n[np.searchsorted(t, grid, side="right") - 1]
    p_sim = np.bincount(X, minlength=6)[:6] / len(X)
    rho = lam_f / (mu_f * (1 - q))
    print(f"q = {q}: input cv {gaps.std()/gaps.mean():.3f}   p_sim {np.round(p_sim, 3)}   geometric {np.round((1-rho)*rho**np.arange(6), 3)}")
# TODO: which theorem is illustrated, and which hypothesis of Burke's theorem does the INPUT violate?

---
## Part D — Prompt log (keep it whenever AI was used)

| # | Where I used AI | Prompt (verbatim or faithful summary) | What it returned | What I verified / corrected |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |

**Reflection (2–3 sentences):** where was the AI most useful, and where was it
least trustworthy, on this homework?

> ...